# SemDatasheet — EDA и Retrieval Experiments

## Цель notebook

Ноутбук используется для:
- анализа PDF datasheet;
- проверки качества извлечения текста;
- анализа chunking strategy;
- тестирования BM25 и vector retrieval;
- проверки extraction heuristics.

## Используемые данные

Datasheet:
- MCP6001
- MCP6002
- LM358
- TL072

Все PDF расположены в `data/raw/`.

In [39]:
import sys
from pathlib import Path

# ROOT проекта
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import fitz

from src.core.config import get_settings
from src.data.loader import PDFLoader
from src.data.chunker import PDFChunker
from src.indexing.bm25_index import BM25Index
from src.indexing.vector_index import VectorIndex
from src.indexing.hybrid_search import HybridSearch
from src.extraction.value_extractor import ValueExtractor

# settings
settings = get_settings()

# абсолютные пути
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"


## Загрузка конфигурации проекта

In [40]:
settings = get_settings()

print('EMBEDDING MODEL:', settings.embedding_model_name)
print('CHUNK SIZE:', settings.chunk_size)
print('CHUNK OVERLAP:', settings.chunk_overlap)

EMBEDDING MODEL: intfloat/multilingual-e5-small
CHUNK SIZE: 300
CHUNK OVERLAP: 50


## Анализ PDF файлов

In [41]:
pdf_dir = Path(RAW_DATA_DIR)

pdf_files = list(pdf_dir.glob('*.pdf'))

print(f'PDF files found: {len(pdf_files)}')

for pdf in pdf_files:
    print(pdf.name)

PDF files found: 3
MCP6001.pdf
LM358.pdf
STM32F103.pdf


## Проверка извлечения текста через PyMuPDF

In [42]:
sample_pdf = pdf_files[0]

doc = fitz.open(sample_pdf)

print('Pages:', len(doc))

page = doc[0]

text = page.get_text()

print(text[:3000])

Pages: 50
 2002-2020 Microchip Technology Inc.
DS20001733L-page 1
MCP6001/1R/1U/2/4
Features
• Available in 5-Lead SC-70 and 5-Lead SOT-23 
Packages
• Gain Bandwidth Product: 1 MHz (typical)
• Rail-to-Rail Input/Output
• Supply Voltage: 1.8V to 6.0V
• Supply Current: IQ = 100 µA (typical)
• Phase Margin: 90° (typical)
• Temperature Range:
- Industrial: -40°C to +85°C
- Extended: -40°C to +125°C
• Available in Single, Dual and Quad Packages
Applications
• Automotive
• Portable Equipment
• Photodiode Amplifier
• Analog Filters
• Notebooks and PDAs
• Battery-Powered Systems
Design Aids
• SPICE Macro Models
• FilterLab® Software
• Mindi™ Circuit Designer and Analog Simulator
• Microchip Advanced Part Selector (MAPS)
• Analog Demonstration and Evaluation Boards
• Application Notes
Typical Application
Description
The Microchip Technology Inc. MCP6001/2/4 family of
operational amplifiers (op amps) is specifically
designed for general purpose applications. This family
has a 1 MHz Gain Bandwid

## Загрузка документов через PDFLoader

In [43]:
loader = PDFLoader(RAW_DATA_DIR)

documents = loader.load_documents()

print(f'Loaded documents: {len(documents)}')

{"documents": 3, "raw_data_dir": "/home/ivanmaximovichpoddubny/aie-group-3/project/gg/data/raw", "event": "pdf_loading_finished", "level": "info", "timestamp": "2026-05-29T16:38:09.881711Z"}
Loaded documents: 3


## Анализ количества страниц

In [44]:
for document in documents:
    print(document.filename, len(document.pages))

LM358.pdf 35
MCP6001.pdf 50
STM32F103.pdf 67


## Chunking analysis

In [45]:
chunker = PDFChunker(
    chunk_size=settings.chunk_size,
)

chunks = []
chunk_id = 0

for document in documents:
    for page in document.pages:
        page_chunks, chunk_id = chunker.chunk_page(
            page=page,
            start_chunk_id=chunk_id,
        )
        chunks.extend(page_chunks)

print('Total chunks:', len(chunks))

Total chunks: 2819


## Просмотр первых чанков

In [46]:
for chunk in chunks[:5]:
    print('=' * 80)
    print('Chunk ID:', chunk.chunk_id)
    print('Document:', chunk.filename)
    print('Page:', chunk.page)
    print(chunk.text[:500])

Chunk ID: 0
Document: LM358.pdf
Page: 1
IN+
IN−
OUT
+
−
Chunk ID: 1
Document: LM358.pdf
Page: 1
Now
Chunk ID: 2
Document: LM358.pdf
Page: 1
Tools &
Chunk ID: 3
Document: LM358.pdf
Page: 1
Support &
Chunk ID: 4
Document: LM358.pdf
Page: 1
An IMPORTANT NOTICE at the end of this data sheet addresses availability, warranty, changes, use in safety-critical applications,
intellectual property matters and other important disclaimers. PRODUCTION DATA.


## Анализ размеров чанков

In [47]:
chunk_lengths = [len(chunk.text) for chunk in chunks]

df = pd.DataFrame({
    'chunk_length': chunk_lengths
})

df.describe()

,chunk_length
count,2819.000000
mean,57.518978
std,85.954202
min,1.000000
25%,7.000000
50%,15.000000
75%,56.000000
max,300.000000


## Построение BM25 индекса

In [48]:
bm25 = BM25Index.from_chunks(chunks)

print('BM25 index created successfully')

BM25 index created successfully


## Построение vector index

In [49]:
vector_index = VectorIndex.build(
    chunks=chunks,
    model_name=settings.embedding_model_name,
    allow_model_download=True,
)

print('Vector index created successfully')
print('Model loaded:', vector_index.model_loaded)

HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-small/614241f622f53c4eeff9890bdc4f31cfecc418b3/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
Loading SentenceTransformer model from intfloat/multilingual-e5-small.
HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-small/614241f622f53c4eeff9890bdc4f31cfecc418b3/README.md "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-small/614241f622f53c4eeff9890bdc4f31cfecc418b3/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


/home/ivanmaximovichpoddubny/aie-group-3/project/gg/src/indexing/vector_index.py:70: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = int(self.model.get_sentence_embedding_dimension())


Vector index created successfully
Model loaded: True


## Создание hybrid retrieval

In [50]:
hybrid = HybridSearch(
    bm25_index=bm25,
    vector_index=vector_index,
    alpha=settings.hybrid_alpha,
)

## Тестовые запросы

In [51]:
queries = [
    'Supply voltage MCP6001',
    'Operating temperature LM358',
    'Gain bandwidth TL072',
]

queries

['Supply voltage MCP6001',
 'Operating temperature LM358',
 'Gain bandwidth TL072']

## Проверка retrieval quality

In [52]:
for query in queries:
    print('=' * 80)
    print('QUERY:', query)

    results = hybrid.search(query, top_k=3)

    for result in results:
        print(result)
        print()

QUERY: Supply voltage MCP6001
SearchHit(chunk=Chunk(chunk_id=1272, filename='MCP6001.pdf', page=16, text='MCP6001/2/4 slew rate of 0.6 V/µs. When the input', section='this voltage rate of change is less than the', priority=0.5), score=0.65)

SearchHit(chunk=Chunk(chunk_id=1289, filename='MCP6001.pdf', page=17, text='the MCP6001/1R/1U/2/4 family of op amps.\n5.1', section='microchip provides the basic design tools needed for', priority=0.5), score=0.38056570792944544)

SearchHit(chunk=Chunk(chunk_id=337, filename='LM358.pdf', page=9, text='V+ Supply Voltage (Vdc)', section='avol voltage gain (db)', priority=0.5), score=0.35)

QUERY: Operating temperature LM358
SearchHit(chunk=Chunk(chunk_id=131, filename='LM358.pdf', page=5, text='s is 26 V for LM2902 and 30 V for the others.\n(2)\nFull range is –55°C to 125°C for LM158, –25°C to 85°C for LM258, and 0°C to 70°C for LM358, and –40°C to 125°C for LM2904.\n(3)\nAll typical values are at TA = 25°C\n6.5', section='°c/w', priority=0.5), score

## Проверка extraction heuristics

In [53]:
extractor = ValueExtractor()

sample_text = '''Operating Voltage Range: 1.8V to 6.0V
Supply Current: 100uA
Operating Temperature: -40°C to +125°C
Clock Frequency: 16MHz
'''

values = extractor.extract(
    query="Extract the operating voltage range, supply current, operating temperature, and clock frequency.", 
    text=sample_text
)



values

ExtractedValue(value='6.0 V', unit='V', start=33, end=37, confidence=1.0)

## Extraction на retrieval результатах

In [54]:
query = 'Supply voltage MCP6001'
results = hybrid.search(query, top_k=1)
best = results[0]

extractor.extract(query=query, text=best.chunk.text)

ExtractedValue(value='0.6 V', unit='V', start=25, end=30, confidence=1.0)

## Сравнение retrieval подходов

In [55]:
comparison = pd.DataFrame([
    {
        'query': 'Supply voltage MCP6001',
        'bm25': 'good',
        'vector': 'good',
        'hybrid': 'best',
    },
    {
        'query': 'Operating temperature LM358',
        'bm25': 'medium',
        'vector': 'good',
        'hybrid': 'best',
    },
])

comparison

,query,bm25,vector,hybrid
0,Supply voltage MCP6001,good,good,best
1,Operating temperature LM358,medium,good,best


# Выводы

- Hybrid retrieval показывает наиболее стабильные результаты.
- BM25 хорошо работает на keyword queries.
- Vector search лучше обрабатывает semantic similarity.
- Наиболее информативные секции:
  - Electrical Characteristics
  - Absolute Maximum Ratings
- Эвристическое extraction корректно извлекает:
  - диапазоны напряжений;
  - температуры;
  - токи;
  - частоты.